# Market Snapshots Edge Research (Intradía)

Este notebook analiza **snapshots tipo Finviz** guardados en **SQLite**, construye features por `ticker`/`category`, genera un dataset de *trades* de investigación, y evalúa señales para anticipar **momentum intradía**.

**Contenido**

- Setup e imports
- Carga SQLite y normalización de esquema
- Limpieza: `timestamp`, `volume`, `change_pct`
- Features por ticker/categoría: streak, persistencia, momentum
- Dataset de trades (plantilla)
- Métricas por segmento y baseline ML (opcional)

> Nota: es un *research notebook*, no un backtest de ejecución real (slippage/latencia/fees no modelados).


## 0. Setup

In [ ]:
# Imports
# Si necesitas paquetes extra, instala aquí:
# %pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm


## 1. Config

In [ ]:
# --- Config del usuario ---
import os

# RUTAS AUTOMÁTICAS DEL PROYECTO
POSSIBLE_PATHS = [
    '../python/finviz_snapshots.db',
    '../finviz_snapshots.db',
    '../snapshots.sqlite',
    './snapshots.sqlite',
    '../python/snapshots.sqlite'
]

DB_PATH = None
found_path = None

print('🔍 Buscando base de datos:')
for i, p in enumerate(POSSIBLE_PATHS, 1):
    check = Path(p)
    status = '✅ ENCONTRADA' if check.exists() else '❌ No existe'
    print(f'   {i}. {p} -> {status}')
    
    if check.exists() and found_path is None:
        found_path = p

print(f'\n📂 Directorio notebook: {os.getcwd()}')

assert found_path is not None, '\n\n❌ NO SE ENCONTRÓ LA BASE DE DATOS. Pon la ruta manualmente en DB_PATH.'

DB_PATH = found_path
print(f'\n✅ BASE DE DATOS CARGADA: {DB_PATH}')
SNAPS_TABLE = None             # si sabes el nombre ponlo, si no, se autodetecta

# Columnas esperadas (se mapean si tu esquema usa otros nombres)
COLMAP_CANDIDATES = {
    'timestamp': ['timestamp','ts','time','datetime','date_time','created_at'],
    'ticker': ['ticker','symbol'],
    'category': ['category','cat','group','sector','theme'],
    'price': ['price','last','last_price'],
    'change_pct': ['change_pct','change','pct_change','changePercent','chg_pct'],
    'volume': ['volume','vol','total_volume']
}


## 2. Load snapshots from SQLite

In [ ]:
# Conecta y autodetecta la tabla si no se especifica
assert Path(DB_PATH).exists(), 'No encuentro DB_PATH: ' + str(DB_PATH)

con = sqlite3.connect(DB_PATH)

if SNAPS_TABLE is None:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    print(tables)
    assert len(tables) > 0, 'La base de datos no tiene tablas.'
    SNAPS_TABLE = tables['name'].iloc[0]

print('Using table:', SNAPS_TABLE)

snaps_raw = pd.read_sql('SELECT * FROM ' + SNAPS_TABLE, con)
print(snaps_raw.head())
print('rows:', len(snaps_raw), 'cols:', snaps_raw.shape[1])


## 3. Normalize schema

In [ ]:
# Mapea columnas a nombres estándar si es posible
snaps = snaps_raw.copy()

lower_cols = {c: c.lower() for c in snaps.columns}

mapped = {}
for std_col, candidates in COLMAP_CANDIDATES.items():
    found = None
    for cand in candidates:
        for c in snaps.columns:
            if lower_cols[c] == cand.lower():
                found = c
                break
        if found is not None:
            break
    if found is not None:
        mapped[found] = std_col

snaps = snaps.rename(columns=mapped)

required = ['timestamp','ticker']
for rc in required:
    assert rc in snaps.columns, 'Falta columna requerida: ' + rc + ' (tus columnas: ' + str(list(snaps.columns)) + ')'

# Si faltan algunas opcionales, las creamos como NaN
for oc in ['category','price','change_pct','volume']:
    if oc not in snaps.columns:
        snaps[oc] = np.nan

print(snaps.head())
print(snaps[['timestamp','ticker','category','price','change_pct','volume']].head())


## 4. Clean fields

In [ ]:
# Timestamp
snaps['timestamp'] = pd.to_datetime(snaps['timestamp'], errors='coerce', utc=True)

# Numeric cleaning
snaps['price'] = pd.to_numeric(snaps['price'], errors='coerce')

# change_pct: soporta strings tipo '3.4%' o '3.4'
def parse_pct(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        x2 = x.replace('%','').replace(',','').strip()
        if x2 == '':
            return np.nan
        return pd.to_numeric(x2, errors='coerce')
    return pd.to_numeric(x, errors='coerce')

snaps['change_pct'] = snaps['change_pct'].apply(parse_pct)

# volume: soporta '1.2M', '450K'
def parse_vol(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.replace(',','').strip().upper()
        if s.endswith('M'):
            return pd.to_numeric(s[:-1], errors='coerce') * 1e6
        if s.endswith('K'):
            return pd.to_numeric(s[:-1], errors='coerce') * 1e3
        if s == '':
            return np.nan
        return pd.to_numeric(s, errors='coerce')
    return pd.to_numeric(x, errors='coerce')

snaps['volume'] = snaps['volume'].apply(parse_vol)

# Basic filter
snaps = snaps.dropna(subset=['timestamp','ticker']).copy()

# Round timestamp to minute bucket for grouping
snaps['snapshot_min'] = snaps['timestamp'].dt.floor('min')

print(snaps.head())
print(snaps.describe(include='all').T.head(20))


## 5. Feature engineering

In [ ]:
snaps_1 = snaps.sort_values(['ticker','snapshot_min']).reset_index(drop=True)

# lag features
snaps_1['chg_lag1'] = snaps_1.groupby('ticker')['change_pct'].shift(1)
snaps_1['chg_delta'] = snaps_1['change_pct'] - snaps_1['chg_lag1']

# simple momentum proxies
snaps_1['chg_roll3_mean'] = snaps_1.groupby('ticker')['change_pct'].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
snaps_1['chg_roll6_mean'] = snaps_1.groupby('ticker')['change_pct'].rolling(6, min_periods=1).mean().reset_index(level=0, drop=True)

# streaks: consecutive presence in snapshots (by minute)
# define a break if gap > 1 minute
prev_min = snaps_1.groupby('ticker')['snapshot_min'].shift(1)
gap_min = (snaps_1['snapshot_min'] - prev_min).dt.total_seconds() / 60.0
new_run = (gap_min.isna()) | (gap_min > 1.0)
run_id = new_run.groupby(snaps_1['ticker']).cumsum()

snaps_1['run_id'] = run_id
snaps_1['streak_len'] = snaps_1.groupby(['ticker','run_id']).cumcount() + 1

# category change
snaps_1['cat_prev'] = snaps_1.groupby('ticker')['category'].shift(1)
snaps_1['cat_changed'] = (snaps_1['category'] != snaps_1['cat_prev']).astype(int)

# time-of-day
snaps_1['hour'] = snaps_1['snapshot_min'].dt.hour
snaps_1['minute'] = snaps_1['snapshot_min'].dt.minute

print(snaps_1.head(20))


## 6. Trades dataset (plantilla research)

In [ ]:
# Construcción simple de trades basados en streak
MIN_STREAK_FOR_ENTRY = 3
HOLD_N_SNAPS = 6

trade_rows = []

for tkr, g in tqdm(snaps_1.groupby('ticker', sort=False), total=snaps_1['ticker'].nunique()):
    g = g.reset_index(drop=True)
    for i in range(len(g)):
        if g.loc[i, 'streak_len'] == MIN_STREAK_FOR_ENTRY:
            entry_ix = i
            exit_ix = min(i + HOLD_N_SNAPS, len(g) - 1)
            entry_price = g.loc[entry_ix, 'price']
            exit_price = g.loc[exit_ix, 'price']
            if pd.isna(entry_price) or pd.isna(exit_price) or entry_price <= 0:
                continue
            trade_ret = (exit_price / entry_price) - 1
            trade_rows.append({
                'ticker': tkr,
                'entry_ts': g.loc[entry_ix, 'snapshot_min'],
                'exit_ts': g.loc[exit_ix, 'snapshot_min'],
                'entry_price': float(entry_price),
                'exit_price': float(exit_price),
                'entry_chg': float(g.loc[entry_ix, 'change_pct']) if pd.notna(g.loc[entry_ix, 'change_pct']) else np.nan,
                'exit_chg': float(g.loc[exit_ix, 'change_pct']) if pd.notna(g.loc[exit_ix, 'change_pct']) else np.nan,
                'entry_vol': float(g.loc[entry_ix, 'volume']) if pd.notna(g.loc[entry_ix, 'volume']) else np.nan,
                'duration_min': (g.loc[exit_ix, 'snapshot_min'] - g.loc[entry_ix, 'snapshot_min']).total_seconds() / 60.0,
                'n_snaps': int(exit_ix - entry_ix + 1),
                'entry_hour': int(g.loc[entry_ix, 'hour']),
                'entry_category': g.loc[entry_ix, 'category'],
                'trade_ret': float(trade_ret)
            })

trades = pd.DataFrame(trade_rows)
print(trades.head())
print('n_trades:', len(trades))


## 7. Edge by category / hour

In [ ]:
if len(trades) > 0:
    agg_cat = trades.groupby('entry_category').agg(
        n=('trade_ret','size'),
        mean_ret=('trade_ret','mean'),
        med_ret=('trade_ret','median'),
        win_rate=('trade_ret', lambda x: float((x>0).mean()))
    ).sort_values('n', ascending=False)

    print(agg_cat.head(20))

    plt.figure(figsize=(10,4))
    tmp = agg_cat.reset_index().head(20)
    sns.barplot(data=tmp, x='entry_category', y='mean_ret', color='#2E86AB')
    plt.xticks(rotation=45, ha='right')
    plt.title('Mean trade return by category (top 20 by n)')
    plt.tight_layout()
    plt.show()

    agg_hour = trades.groupby('entry_hour').agg(n=('trade_ret','size'), mean_ret=('trade_ret','mean'))
    print(agg_hour)

    plt.figure(figsize=(8,3))
    sns.lineplot(data=agg_hour.reset_index(), x='entry_hour', y='mean_ret', marker='o', color='#C0392B')
    plt.title('Mean trade return by entry hour')
    plt.tight_layout()
    plt.show()


## 8. Baseline model (opcional)

In [ ]:
# Dataset de clasificación: predecir si trade_ret > 0 usando features del snapshot de entrada
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

if len(trades) > 200:
    # reconstruye features del entry snapshot uniendo por ticker+entry_ts
    entry_feats = snaps_1[['ticker','snapshot_min','change_pct','chg_delta','chg_roll3_mean','chg_roll6_mean','volume','cat_changed','streak_len','hour']].copy()
    entry_feats = entry_feats.rename(columns={'snapshot_min':'entry_ts'})

    ds = trades.merge(entry_feats, on=['ticker','entry_ts'], how='left')
    ds['y'] = (ds['trade_ret'] > 0).astype(int)

    feat_cols = ['change_pct','chg_delta','chg_roll3_mean','chg_roll6_mean','volume','cat_changed','streak_len','hour']
    X = ds[feat_cols].copy()
    y = ds['y'].copy()

    # fill
    X = X.replace([np.inf,-np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    tss = TimeSeriesSplit(n_splits=5)
    aucs = []
    aps = []

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=200))
    ])

    for tr_idx, te_idx in tss.split(X):
        X_tr = X.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        X_te = X.iloc[te_idx]
        y_te = y.iloc[te_idx]

        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:,1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))

    print('AUC mean:', float(np.mean(aucs)))
    print('AP  mean:', float(np.mean(aps)))

else:
    print('Pocas trades para ML. Necesitas más historial o ajustar reglas para generar más entradas.')
